# AI Simple RAG Implementation

## Importing the libraries

In [1]:
import os 
import json
import gradio as gr
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

## Loading OpenAI API Key

In [2]:
load_dotenv(override = True)
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("No API key found!")
else:
    print("API key found!")

API key found!


## asdasd

In [3]:
knowledge = {
    "employees": {},
    "products": {},
    "company": {},
    "contracts": {}
}

## asdsad

In [4]:
for file in Path("knowledge-base/employees").glob("*"):
    name = file.stem.lower()
    with file.open("r", encoding = "utf-8") as f:
        knowledge["employees"][name] = f.read()

## asdsad

In [5]:
for file in Path("knowledge-base/products").glob("*"):
    name = file.stem.lower()
    with file.open("r", encoding = "utf-8") as f:
        knowledge["products"][name] = f.read()

## asdasd

In [6]:
for file in Path("knowledge-base/company").glob("*"):
    name = file.stem.lower()
    with file.open("r", encoding = "utf-8") as f:
        knowledge["company"][name] = f.read()

## adsads

In [7]:
for file in Path("knowledge-base/contracts").glob("*"):
    name = file.stem.lower()
    with file.open("r", encoding = "utf-8") as f:
        knowledge["contracts"][name] = f.read()

## asdasd

In [ ]:
system_prompt = """
You are an AI Assistant for InsureLLM (An Insurance Company)

Your job is to answer questions only about:
- InsureLLM employees
- InsureLLM products
- InsureLLM company information
- InsureLLM contracts

Rules:
- Reply to greetings.
- Do not explicitly mention employees, products, company information, and contracts.
- Use the provided context whenever possible.
- If the answer is not in the context, say you do not know.
- Do not answer unrelated questions.
- Do not provide programming help, general knowledge, or personal advice.

Relevant context:
"""

## asdasd

In [40]:
def get_context(keyword, category):

    keyword = keyword.lower()
    
    if category.lower() == "employees":
        return [knowledge["employees"][employee] for employee in knowledge["employees"] if any(item in keyword.split() for item in employee.split())]
 
    elif category.lower() == "products":
        return [knowledge["products"][product] for product in knowledge["products"] if any(item in keyword.split() for item in product.split())]

    elif category.lower() == "company":
        return [knowledge["company"][company] for company in knowledge["company"] if any(item in keyword.split() for item in company.split())]

    elif category.lower() == "contracts":
        return [knowledge["contracts"][contract] for contract in knowledge["contracts"] if any(item in keyword.split() for item in contract.split())]
    
    else:
        return["Sorry but there is no additional context relevant to the user's question."]

In [41]:
get_context_desc = {
    "name": "get_context",
    "description": "Searches the Insurellm internal knowledge base for relevant information about employees, products, company information, or contracts.",
    "parameters": {
        "type": "object",
        "properties": {
            "keyword": {
                "type": "string",
                "description": "The specific entity or topic to search for. For example: an employee name ('Michael O'Brien'), a product name ('CarLLM'), a company topic, or a contract name."
            },
            "category": {
                "type": "string",
                "enum": [
                    "employees",
                    "products",
                    "company",
                    "contracts"
                ],
                "description": "The knowledge-base category where the information should be searched."
            }
        },
        "required": ["keyword", "category"],
        "additionalProperties": False
    }
}

In [42]:
tools = [{
    "type": "function",
    "function": get_context_desc
}]

## asdasd

In [43]:
client = OpenAI()

In [44]:
def handle_tool_calls(message):

    responses = []
    for tool_call in message.tool_calls:

        if tool_call.function.name == "get_context":
            
            arguments = json.loads(tool_call.function.arguments)
            keyword = arguments.get("keyword")
            category = arguments.get("category")
            context = get_context(keyword, category)

            responses.append({
                "role": "tool",
                "content": "\n\n".join(context),
                "tool_call_id": tool_call.id
            })

    return responses

## asdasd

In [45]:
def chat(message, history):

    history = [{
        "role": h["role"],
        "content": h["content"]
    } for h in history ]

    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = client.chat.completions.create(
        model = "gpt-5.4-mini",
        messages = messages,
        tools = tools
    )

    while response.choices[0].message.tool_calls:

        message = response.choices[0].message
        tool_response = handle_tool_calls(message)
        messages.append(message)
        messages.extend(tool_response)
        response = client.chat.completions.create(
            model = "gpt-5.4-mini",
            messages = messages
        )
    
    return response.choices[0].message.content

## asdsad

In [46]:
interface = gr.ChatInterface(
    fn = chat
)

In [49]:
interface.launch(
    inbrowser = True
)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [48]:
interface.close()

Closing server running on port: 7862
